# ARC2 NVARC+ v1 — test-time-trained Qwen3-4B (NVARC 2025) with adaptive budget use

Core = the open-sourced ARC Prize 2025 winner (NVARC: `qwen3_4b_grids15_sft139` + per-task LoRA
test-time training + batched DFS decoding + augmentation re-scoring + `score_kgmon` selection),
with the throughput patch from the public LB 33.89 lineage, running unchanged as **phase 1**.

Additions in this notebook:
1. **Robustness**: model-path resolution, deterministic scoring seeds, `None`-sentinel queue loop,
   per-task exception containment, worker restart, corrupt-shard tolerance, identity fallbacks,
   schema re-validation.
2. **Cheap-first task order**: the 12 h deadline truncates the most expensive tail instead of a
   random alphabetical tail.
3. **Phase 2 (adaptive)**: leftover time goes to (a) tasks never reached and (b) outputs with
   fewer than two distinct candidates, which get a wider search (new seeds, 24 views, DFS
   threshold 0.1). Candidates are pooled and vote together; phase-1 results are never changed.

Interactive/commit runs use a 55-minute budget on 4 debug evaluation tasks so every phase is
exercised cheaply; the hidden-test rerun uses 12 h − 10 min.


In [1]:
# ---------------------------------------------------------------------------
# Global wall-clock budget. Rerun (hidden test): 12h minus a 10 min write buffer.
# ---------------------------------------------------------------------------
import os, time, json
RERUN = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))
T0 = time.time()
if RERUN:
    global_end_time = T0 + 12 * 3600 - 600
elif os.path.exists("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"):
    # If test file is available in interactive mode, allocate 9 hours minus 10m buffer
    global_end_time = T0 + 9 * 3600 - 600
else:
    global_end_time = T0 + 4 * 60  # Quick 4-min sanity commit
print(f"rerun={RERUN} budget={(global_end_time-T0)/3600:.2f}h cutoff={time.ctime(global_end_time)}")


rerun=False budget=0.92h


In [2]:
!pip uninstall -y tensorflow torchao
import torch, sys, os, glob

# Auto-discover or insert utility scripts
for p in glob.glob("/kaggle/usr/lib/**", recursive=True) + glob.glob("/kaggle/input/**/unsloth", recursive=True):
    if os.path.isdir(p) and os.path.isfile(os.path.join(p, "unsloth", "__init__.py")):
        if p not in sys.path:
            sys.path.insert(0, p)
            print(f"[Setup] Loaded Unsloth from {p}")
            break

print(sys.version.split()[0], "torch", torch.__version__, "gpus", torch.cuda.device_count(),
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print("models:", glob.glob("/kaggle/input/models/*/*"), glob.glob("/kaggle/input/qwen*"))
os.makedirs("/kaggle/working/logs", exist_ok=True)
os.makedirs("/kaggle/inference_outputs", exist_ok=True)
os.makedirs("/kaggle/inference_outputs_deep", exist_ok=True)


Found existing installation: tensorflow 2.18.0
Uninstalling tensorflow-2.18.0:
  Successfully uninstalled tensorflow-2.18.0
3.11.13 torch 2.8.0+cu128 gpus 4 ['NVIDIA L4', 'NVIDIA L4', 'NVIDIA L4', 'NVIDIA L4']
models: ['/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139'] []
utility scripts: ['/kaggle/usr/lib', '/kaggle/usr/lib/notebooks/sorokin/pip_install_unsloth_flash_patch', '/kaggle/usr/lib/notebooks/sorokin/pip_install_unsloth_flash_patch/setuptools/_vendor']


In [3]:
%%writefile arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    return isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation)==list(range(10))
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except:
            pass
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = g.tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score

Writing arc_loader.py


In [4]:
%%writefile arc_decoder.py
import os
import bz2
import pickle
import numpy as np

def hashable(guess):
    return tuple(map(tuple, guess))

def score_sum(guesses, getter):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)
    scores = [(getter(sc), o) for sc, o in scores.values()]
    scores = sorted(scores, key=(lambda x: x[0]), reverse=True)
    ordered_outputs = [x[-1] for x in scores]
    return ordered_outputs

def getter_full_probmul_3(guesses, baseline=3):
    inf_score = np.sum([baseline-g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline-s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)


selection_algorithms = [
    score_full_probmul_3,
    score_kgmon,
]


def _valid_sample(sample):
    try:
        sol = np.asarray(sample["solution"])
        if sol.ndim != 2 or sol.size == 0 or sol.shape[0] > 30 or sol.shape[1] > 30:
            return False
        if not np.issubdtype(sol.dtype, np.integer):
            return False
        if sol.min() < 0 or sol.max() > 9:
            return False
        if not np.isfinite(sample["beam_score"]):
            return False
        if not len(sample["score_aug"]) or not np.all(np.isfinite(sample["score_aug"])):
            return False
        return True
    except Exception:
        return False


class ArcDecoder:

    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}
        try:
            self.valid_keys = set(dataset.keys)
        except Exception:
            self.valid_keys = None

    def load_decoded_results(self, store, run_name=""):
        if not os.path.isdir(store):
            print(f"*** No decoded results at {store}")
            return 0
        n_files = n_samples = n_bad = 0
        for key in os.listdir(store):
            try:
                with bz2.BZ2File(os.path.join(store, key)) as f:
                    outputs = pickle.load(f)
            except Exception as e:
                print(f"*** Skipping corrupt shard {key}: {e}")
                n_bad += 1
                continue
            n_files += 1
            base_key = key.split(".")[0]
            if self.valid_keys is not None and base_key not in self.valid_keys:
                n_bad += 1
                continue
            for i, sample in enumerate(outputs):
                if not _valid_sample(sample):
                    n_bad += 1
                    continue
                self.decoded_results.setdefault(base_key, {})[f"{key}{run_name}.out{i}"] = sample
                n_samples += 1
        print(f"*** Loaded {n_files} shards / {n_samples} samples from {store} (skipped {n_bad})")
        return n_samples

    def candidate_stats(self):
        """Per output key: number of unique candidate grids and number of samples."""
        stats = {}
        for bk, v in self.decoded_results.items():
            uniq = {hashable(g["solution"]) for g in v.values()}
            stats[bk] = dict(unique=len(uniq), samples=len(v))
        return stats

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        return {bk: selection_algorithm({k: g for k, g in v.items()}) for bk, v in self.decoded_results.items()}

    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms...")

        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0

        correct_beam_scores = []

        for basekey, basevalues in self.decoded_results.items():

            mult_key, mult_sub = basekey.split("_")
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)

            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]

            for subkey, sample in basevalues.items():

                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])

                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"

                output_len = f"{solution.shape[0]}x{solution.shape[1]}"

                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1

        print(f" subkeys: {num_solved_keys}/{num_total_keys}")
        if correct_beam_scores:
            print(f" avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
            print(f" max correct beam score: {np.max(correct_beam_scores):8.5f}")

        num_puzzles = len(num_tasks_per_puzzle)

        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            selected = self.run_selection_algo(selection_algorithm)
            correct_puzzles = {k for k, v in selected.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
            print(correct_puzzles)
            score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
            print(f" acc: {score:5.1f}/{num_puzzles:3} ('{name}')")


Writing arc_decoder.py


In [5]:
%%writefile arc_solver.py
import sys, os, glob
# Suppress incompatible torchao version check in transformers
if "torchao" not in sys.modules:
    sys.modules["torchao"] = None

for p in glob.glob("/kaggle/usr/lib/**", recursive=True) + glob.glob("/kaggle/input/**/unsloth", recursive=True):
    if os.path.isdir(p) and os.path.isfile(os.path.join(p, "unsloth", "__init__.py")):
        if p not in sys.path:
            sys.path.insert(0, p)

try:
    from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
    HAS_UNSLOTH = True
except ImportError:
    HAS_UNSLOTH = False
    from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
    from peft import LoraConfig, get_peft_model
    UnslothTrainingArguments = TrainingArguments
    UnslothTrainer = Trainer

from arc_loader import ArcDataset, QwenFormatter, is_valid_solution
from arc_decoder import hashable

import gc
import io
import time
import zlib
import torch
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union
from transformers import DataCollatorForLanguageModeling

import logging
from contextlib import redirect_stdout, redirect_stderr

from peft import get_peft_model_state_dict, set_peft_model_state_dict

import bz2
import pickle
import traceback

logging.disable(logging.WARNING)

def _env(name, default, cast):
    v = os.getenv(name)
    return cast(v) if v not in (None, "") else default

CFG = dict(
    model_path      = _env("ARC_MODEL_PATH", "", str),
    out_dir         = _env("ARC_OUT_DIR", "/kaggle/inference_outputs", str),
    lora_seed       = _env("ARC_LORA_SEED", 42, int),
    train_aug_seed  = _env("ARC_TRAIN_AUG_SEED", 1, int),
    n_train_aug     = _env("ARC_N_TRAIN_AUG", 4, int),
    num_epochs      = _env("ARC_EPOCHS", 1, int),
    learning_rate   = _env("ARC_LR", 5e-5, float),
    eval_aug_seed   = _env("ARC_EVAL_AUG_SEED", 2, int),
    n_eval_aug      = _env("ARC_N_EVAL_AUG", 1, int),
    min_prob        = _env("ARC_MIN_PROB", 0.2, float),
    dfs_window      = _env("ARC_DFS_WINDOW", 180.0, float),
    task_cap        = _env("ARC_TASK_CAP", 360.0, float),
    score_seed_off  = _env("ARC_SCORE_SEED_OFFSET", 0, int),
    decode_batch    = _env("ARC_DECODE_BATCH", 4, int),
)

ARC_VOCAB = {
    "0": 0, "1": 1, "2": 2, "3": 3, "4": 4,
    "5": 5, "6": 6, "7": 7, "8": 8, "9": 9,
    "\n": 10, "<|im_end|>": 11,
}

ARC_TOKENS = [
    15, 16, 17, 18, 19,
    20, 21, 22, 23, 24,
    198, 151645
]

PAD_ID = 151643
EOS_ID = 151645
USER_TOKEN_ID = 872
ASSISTANT_TOKEN_ID = 77091


def resolve_model_dir():
    if CFG["model_path"] and os.path.isdir(CFG["model_path"]):
        return CFG["model_path"]
    candidates = [
        "/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
        "/kaggle/input/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
        "/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/Transformers/bfloat16/1",
        "/kaggle/input/arc-qwen-model",
        "/kaggle/input/qwen-models",
    ]
    for c in candidates:
        if os.path.isfile(os.path.join(c, "config.json")):
            return c
    for c in glob.glob("/kaggle/input/**/config.json", recursive=True):
        d = os.path.dirname(c)
        if os.path.isfile(os.path.join(d, "tokenizer.json")) or os.path.isfile(os.path.join(d, "tokenizer_config.json")):
            return d
    return "/kaggle/input/qwen3_4b_grids15_sft139"


def stable_seed(key, offset=0):
    return (zlib.crc32(key.encode("utf-8")) + offset) % (1024 ** 2)


def make_training_args(**kwargs):
    if HAS_UNSLOTH:
        return UnslothTrainingArguments(**kwargs)
    import inspect
    sig = inspect.signature(TrainingArguments.__init__)
    params = sig.parameters
    if any(param.kind == inspect.Parameter.VAR_KEYWORD for param in params.values()):
        return TrainingArguments(**kwargs)
    cleaned = {k: v for k, v in kwargs.items() if k in params}
    return TrainingArguments(**cleaned)


class UnslothFixedTrainer(UnslothTrainer):

    def __init__(self, *args, **kwargs):
        import inspect
        sig = inspect.signature(UnslothTrainer.__init__)
        params = sig.parameters
        cleaned = {}
        for k, v in kwargs.items():
            if k in params:
                cleaned[k] = v
            elif k == "tokenizer" and "processing_class" in params:
                cleaned["processing_class"] = v
            elif k == "processing_class" and "tokenizer" in params:
                cleaned["tokenizer"] = v
        super().__init__(*args, **cleaned)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        import torch.nn as nn
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if getattr(self.args, "past_index", -1) >= 0:
            self._past = outputs[self.args.past_index]

        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss_fct = nn.CrossEntropyLoss()
                loss = loss_fct(outputs.logits.view(-1, outputs.logits.shape[-1]), labels.view(-1))
            else:
                loss = self.label_smoother(outputs, labels, shift_labels=True)
        else:
            if isinstance(outputs, dict) and "loss" not in outputs:
                raise ValueError("The model did not return a loss from inputs.")
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]

        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):

    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            user_start_idx = np.where(labels == USER_TOKEN_ID)[0].tolist()
            assistant_start_idx = np.where(labels == ASSISTANT_TOKEN_ID)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            end_idx = np.where(labels == EOS_ID)[0]
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                assert start < end
                if j % 2 == 1:
                    start += 2
                    end += 1
                    batch["labels"][i, start:end] = labels[start:end]
        return batch


_ARC_TOKEN_ID_CACHE = {}


def _arc_token_ids(device):
    key = str(device)
    token_ids = _ARC_TOKEN_ID_CACHE.get(key)
    if token_ids is None:
        token_ids = torch.tensor(ARC_TOKENS, dtype=torch.long, device=device)
        _ARC_TOKEN_ID_CACHE[key] = token_ids
    return token_ids


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time, dfs_window) -> dict:
    n = logits.size(0)
    logits_f = logits.float()
    token_ids = _arc_token_ids(logits.device)
    arc_logits = logits_f.index_select(-1, token_ids)
    nll = (
        torch.as_tensor(scores, dtype=torch.float32, device=logits.device).view(n, 1)
        + torch.logsumexp(logits_f, dim=-1, keepdim=True)
        - arc_logits
    ).cpu()

    suffixes = defaultdict(list)
    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for token_idx, t in enumerate(ARC_TOKENS):
            score = nll[i, token_idx].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i] = sorted(candidates[i], key=lambda x:x[0])

    while time.time() - start_time < dfs_window and time.time() < end_time:
        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens-1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos+1,
            cache=outputs.past_key_values,
            start_time=start_time,
            end_time=end_time,
            dfs_window=dfs_window,
        )

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time, dfs_window):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
        dfs_window=dfs_window,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x:x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    max_len = max(batch_lengths)
    padded_tokens = []
    for tokens in batch_tokens:
        padded = tokens + [PAD_ID] * (max_len - len(tokens))
        padded_tokens.append(padded)
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)

    outputs = model(input_ids=input_ids, return_dict=True, use_cache=False)
    batch_logits = outputs.logits.float()
    batch_log_norm = torch.logsumexp(batch_logits, dim=-1)
    result = []
    for row_id, (query_tokens, answer_tokens) in enumerate(zip(batch_query_tokens, batch_answer_tokens)):
        query_length = len(query_tokens)
        answer_length = len(answer_tokens)
        positions = torch.arange(
            query_length - 1,
            query_length - 1 + answer_length,
            device=model.device,
        )
        target_tokens = torch.tensor(answer_tokens, device=model.device, dtype=torch.long)
        answer_log_probs = (
            batch_logits[row_id, positions, target_tokens]
            - batch_log_norm[row_id, positions]
        )
        result.append(-answer_log_probs.sum().item())
    return result


def make_view_batches(eval_ds, n_perm, batch_size):
    test_id_to_subkeys = defaultdict(list)
    for subkey in sorted(eval_ds.keys):
        test_id = subkey.split(".")[0].split("_")[1]
        test_id_to_subkeys[test_id].append(subkey)
    groups_a = [0, 2, 1, 3]
    groups_b = [4, 6, 5, 7]
    batches = []
    for geos in (groups_a, groups_b):
        for test_id, subkeys in test_id_to_subkeys.items():
            if n_perm == 2 and batch_size == 4:
                for a, b in ((geos[0], geos[1]), (geos[2], geos[3])):
                    batches.append(subkeys[a*n_perm:(a+1)*n_perm] + subkeys[b*n_perm:(b+1)*n_perm])
            else:
                views = []
                for g in geos:
                    views.extend(subkeys[g*n_perm:(g+1)*n_perm])
                for i in range(0, len(views), batch_size):
                    batches.append(views[i:i+batch_size])
    return batches


def worker(rank, queue, end_time, test_path=None):
    os.makedirs(CFG["out_dir"], exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    peft_params = dict(
        r=16,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=32,
        lora_dropout=0,
        bias="none",
        random_state=CFG["lora_seed"],
        use_rslora=False,
        loftq_config=None,
    )

    train_args = dict(
        output_dir=f"/kaggle/working/logs/train_logs_rank{rank}",
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=CFG["num_epochs"],
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=CFG["learning_rate"],
        optim="adamw_torch",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=CFG["lora_seed"],
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=not (torch.cuda.is_available() and torch.cuda.is_bf16_supported()),
        bf16=bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported()),
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    max_seq_length = 8192

    model_dir = resolve_model_dir()
    print(f"[Rank {rank}] model dir: {model_dir}")
    print(f"[Rank {rank}] config: {CFG}")

    is_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    compute_dtype = torch.bfloat16 if is_bf16 else torch.float16

    if HAS_UNSLOTH:
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=model_dir,
            full_finetuning=False,
            load_in_4bit=False,
            local_files_only=True,
            use_gradient_checkpointing=False,
            max_seq_length=max_seq_length,
        )
        model = FastLanguageModel.get_peft_model(model, **peft_params)
    else:
        from transformers import AutoTokenizer, AutoModelForCausalLM
        from peft import LoraConfig, get_peft_model
        tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
        model = AutoModelForCausalLM.from_pretrained(
            model_dir,
            torch_dtype=compute_dtype,
            local_files_only=True
        )
        peft_config = LoraConfig(
            r=peft_params.get("r", 16),
            lora_alpha=peft_params.get("lora_alpha", 32),
            target_modules=peft_params.get("target_modules", ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]),
            lora_dropout=peft_params.get("lora_dropout", 0.0),
            bias=peft_params.get("bias", "none"),
            task_type="CAUSAL_LM"
        )
        model = get_peft_model(model, peft_config)

    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(compute_dtype)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer,
        mlm=False,
    )

    formatter = QwenFormatter(tokenizer=tokenizer)

    if test_path is None:
        if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
            test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
        elif os.path.exists("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"):
            test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
        else:
            test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    arc_test_set = ArcDataset.from_file(test_path)

    while time.time() < end_time:
        key = queue.get()
        if key is None:
            break

        out_fname = os.path.join(CFG["out_dir"], f"{key}.pkl.bz2")
        tmp_fname = out_fname + f".tmp.{rank}.{os.getpid()}"
        if os.path.exists(out_fname):
            continue

        try:
            task_start_time = time.time()
            set_peft_model_state_dict(
                model=model,
                peft_model_state_dict=default_weights,
                adapter_name="default",
            )

            model = FastLanguageModel.for_training(model) if HAS_UNSLOTH else model

            puzzle_ds = arc_test_set.change_keys([key])
            train_ds = puzzle_ds.augment(n=CFG["n_train_aug"], shfl_keys=True, seed=CFG["train_aug_seed"])
            train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)

            with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):
                trainer = UnslothFixedTrainer(
                    model=model,
                    tokenizer=tokenizer,
                    data_collator=collator,
                    train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                    dataset_text_field="text",
                    max_seq_length=max_seq_length,
                    args=make_training_args(**train_args),
                )
                stats = trainer.train()
                model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)
                del trainer

            model = FastLanguageModel.for_inference(model) if HAS_UNSLOTH else model
            gc.collect()
            torch.cuda.empty_cache()

            eval_ds = puzzle_ds.augment(n=CFG["n_eval_aug"], shfl_keys=False, seed=CFG["eval_aug_seed"])
            eval_ds_transposed = eval_ds.transpose()
            eval_ds = eval_ds.concat(eval_ds_transposed)

            outputs = {}
            score_queries = {}
            score_answers = {}
            total_guesses = 0

            max_new_tokens = 900
            max_score = 100.0

            batches = make_view_batches(eval_ds, n_perm=CFG["n_eval_aug"], batch_size=CFG["decode_batch"])

            for batch in batches:
                if time.time() - task_start_time > CFG["task_cap"] or time.time() > end_time:
                    break
                batch_eval_ds = eval_ds.change_keys(batch)
                batch_prompts = [formatter.format_query(x) for x in batch_eval_ds.as_list(formatter)]
                prefix_tokens = [tokenizer.encode(p) for p in batch_prompts]

                batch_suffixes = inference_turbo_dfs(
                    model=model,
                    prefix_tokens=prefix_tokens,
                    max_new_tokens=max_new_tokens,
                    max_score=max_score,
                    end_time=end_time,
                    dfs_window=CFG["dfs_window"],
                )

                for batch_id, suffixes in batch_suffixes:
                    subkey = batch[batch_id]
                    test_id = subkey.split(".")[0].split("_")[1]
                    inv_perm = "permute" not in subkey

                    for score, suffix_tokens in suffixes:
                        suffix_text = tokenizer.decode(suffix_tokens)
                        guess = formatter.parse_output(suffix_text)
                        if guess is None:
                            continue

                        inv_guess = eval_ds.invert_mod(guess, subkey, inv_perm=inv_perm)
                        if not is_valid_solution(inv_guess):
                            continue

                        h = hashable(inv_guess)
                        entry = outputs.setdefault(test_id, {}).setdefault(
                            h, {
                                "beam_score": score,
                                "solution": inv_guess,
                                "views": 0,
                                "score_aug": [],
                            }
                        )
                        entry["views"] += 1
                        if score < entry["beam_score"]:
                            entry["beam_score"] = score
                        total_guesses += 1

            for test_id, guesses in outputs.items():
                for h, g in guesses.items():
                    queries = []
                    answers = []
                    score_eval_ds = eval_ds.filter_keys([f"test_{test_id}."])
                    inv_perm = False
                    for subkey in score_eval_ds.keys:
                        fwd_guess = eval_ds.forward_mod(g["solution"], subkey, use_perm=inv_perm)
                        item = score_eval_ds.queries[subkey]
                        queries.append(formatter.format_query(item))
                        answers.append(formatter.format_reply({"output": fwd_guess}))
                    g["score_aug"] = calc_scores(queries, answers, tokenizer, model)

            with bz2.open(tmp_fname, "wb") as f:
                pickle.dump(outputs, f)
            os.replace(tmp_fname, out_fname)

            print(f"[Rank {rank}] finished {key} in {time.time()-task_start_time:.1f}s ({total_guesses} guesses, {len(outputs)} test outputs)")

        except Exception as e:
            print(f"[Rank {rank}] ERROR on puzzle {key}: {type(e).__name__}: {e}")
            traceback.print_exc()
            if os.path.exists(tmp_fname):
                try: os.remove(tmp_fname)
                except Exception: pass

        finally:
            gc.collect()
            torch.cuda.empty_cache()

    print(f"[Rank {rank}] Worker finished.")


Writing arc_solver.py


In [6]:
%%writefile starter.py
import os
import sys
import time
import json
import torch
import argparse
import traceback
import torch.multiprocessing as mp


def local_worker(rank, queue, end_time, test_path, marker_dir):

    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)

    torch.set_default_device("cpu")

    # Serialize the unsloth import/patching across workers (baseline behaviour),
    # but never wait forever for a dead predecessor.
    if rank > 0:
        waited = 0
        while not os.path.exists(os.path.join(marker_dir, f"worker{rank-1}")) and waited < 900:
            time.sleep(5)
            waited += 5

    from arc_solver import worker

    with open(os.path.join(marker_dir, f"worker{rank}"), "w") as f:
        f.write("Ok")

    print(f"[Rank {rank}] start!")

    attempts = 0
    while attempts < 2 and time.time() < end_time:
        attempts += 1
        try:
            worker(rank, queue, end_time, test_path=test_path)
            break
        except Exception as e:
            print(f"[Rank {rank}] worker crashed ({type(e).__name__}: {e}); attempt {attempts}")
            traceback.print_exc()
            try:
                import gc
                gc.collect()
                torch.cuda.empty_cache()
            except Exception:
                pass
            if attempts >= 2:
                print(f"[Rank {rank}] giving up.")

    print(f"[Rank {rank}] done!")


def estimated_work(task):
    """Rough cost proxy: tokens of all train pairs (TTT + context) plus an estimate of
    the decoded output sizes. Cheap tasks first => the deadline truncates the most
    expensive (and least likely to be solved) tail."""
    def ntok(g):
        return len(g) * (len(g[0]) + 1)
    train_tokens = sum(ntok(p["input"]) + ntok(p["output"]) for p in task["train"])
    ratios = [ntok(p["output"]) / max(1, ntok(p["input"])) for p in task["train"]]
    ratios.sort()
    ratio = ratios[len(ratios) // 2]
    test_tokens = sum(ntok(t["input"]) * (1 + ratio) for t in task["test"])
    return train_tokens * 16 + test_tokens * 8 * len(task["test"])


if __name__ == "__main__":

    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, default=0.0)
    parser.add_argument("--keys-file", type=str, default="")
    parser.add_argument("--nprocs", type=int, default=0)
    parser.add_argument("--order", type=str, default="cheap", choices=["cheap", "sorted", "file"])
    parser.add_argument("--test-path", type=str, default="")
    parser.add_argument("--marker-dir", type=str, default="/kaggle/working/markers")
    args, _ = parser.parse_known_args()

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    if args.test_path:
        test_path = args.test_path
    elif rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    with open(test_path, "r") as f:
        data = json.load(f)

    if args.keys_file:
        with open(args.keys_file) as f:
            keys = [k for k in json.load(f) if k in data]
    else:
        keys = sorted(data.keys())
        if not rerun_mode:
            debug_keys = os.getenv("ARC_DEBUG_KEYS", "0934a4d8,36a08778,981571dc,aa4ec2a5").split(",")
            keys = [k for k in keys if k in debug_keys]

    if args.order == "cheap":
        keys = sorted(keys, key=lambda k: estimated_work(data[k]))
    elif args.order == "sorted":
        keys = sorted(keys)

    nprocs = args.nprocs or min(4, torch.cuda.device_count())
    os.makedirs(args.marker_dir, exist_ok=True)
    for f_ in os.listdir(args.marker_dir):
        try:
            os.remove(os.path.join(args.marker_dir, f_))
        except Exception:
            pass

    print(f"[starter] {len(keys)} tasks, {nprocs} workers, order={args.order}, "
          f"budget={(args.end_time - time.time())/60:.1f} min, test_path={test_path}")

    queue = mp.Manager().Queue()
    for key in keys:
        queue.put(key)
    for _ in range(nprocs):
        queue.put(None)

    try:
        mp.spawn(local_worker, args=(queue, args.end_time, test_path, args.marker_dir), nprocs=nprocs)
    except Exception as e:
        print(f"[starter] spawn finished with error: {type(e).__name__}: {e}")
        traceback.print_exc()
    print("[starter] finished.")


Writing starter.py


In [7]:
# ---------------------------------------------------------------------------
# Phase 1 — primary pass: LB-proven NVARC configuration (perfpatch), cheap-first
# task order, 4 workers (one per L4). Gets the entire remaining budget.
# ---------------------------------------------------------------------------
import subprocess, sys, time, os
os.environ.update({
    "UNSLOTH_DISABLE_STATISTICS": "1",
    "TRITON_PTXAS_PATH": "/usr/local/cuda/bin/ptxas",
    "OMP_NUM_THREADS": "12",
    "PYTHONHASHSEED": "0",
    "ARC_OUT_DIR": "/kaggle/inference_outputs",
})
phase1_start = time.time()
rc = subprocess.call([sys.executable, "starter.py", "--end-time", f"{global_end_time}", "--order", "cheap"])
print(f"phase-1 rc={rc} took {(time.time()-phase1_start)/60:.1f} min; remaining {(global_end_time-time.time())/60:.1f} min")


[starter] 4 tasks, 4 workers, order=cheap, budget=54.3 min, test_path=/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
[Rank 0] start!
[Rank 0] model dir: /kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1
[Rank 0] config: {'model_path': '', 'out_dir': '/kaggle/inference_outputs', 'lora_seed': 42, 'train_aug_seed': 1, 'n_train_aug': 16, 'num_epochs': 1, 'learning_rate': 5e-05, 'eval_aug_seed': 2, 'n_eval_aug': 2, 'min_prob': 0.2, 'dfs_window': 540.0, 'task_cap': 1200.0, 'score_seed_off': 0, 'decode_batch': 4}
==((====))==  Unsloth 2025.9.7: Fast Qwen3 patching. Transformers: 4.55.4.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.

Loading checkpoint shards: 100%|██████████| 2/2 [00:43<00:00, 21.82s/it]


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
[Rank 1] start!
[Rank 1] model dir: /kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1
[Rank 1] config: {'model_path': '', 'out_dir': '/kaggle/inference_outputs', 'lora_seed': 42, 'train_aug_seed': 1, 'n_train_aug': 16, 'num_epochs': 1, 'learning_rate': 5e-05, 'eval_aug_seed': 2, 'n_eval_aug': 2, 'min_prob': 0.2, 'dfs_window': 540.0, 'task_cap': 1200.0, 'score_seed_off': 0, 'decode_batch': 4}
==((====))==  Unsloth 2025.9.7: Fast Qwen3 patching. Transformers: 4.55.4.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which ar

Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.10s/it]


Unsloth: Training embed_tokens in mixed precision to save VRAM
Unsloth: Training lm_head in mixed precision to save VRAM
Unsloth: Training embed_tokens in mixed precision to save VRAM
Unsloth: Training lm_head in mixed precision to save VRAM
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
[Rank 2] start!
[Rank 2] model dir: /kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1
[Rank 2] config: {'model_path': '', 'out_dir': '/kaggle/inference_outputs', 'lora_seed': 42, 'train_aug_seed': 1, 'n_train_aug': 16, 'num_epochs': 1, 'learning_rate': 5e-05, 'eval_aug_seed': 2, 'n_eval_aug': 2, 'min_prob': 0.2, 'dfs_window': 540.0, 'task_cap': 1200.0, 'score_seed_off': 0, 'decode_batch': 4}
==((====))==  Unsloth 2025.9.7: Fast Qwen3 patching. Transformers: 4.55.4.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.9.

Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.03s/it]


Unsloth: Training embed_tokens in mixed precision to save VRAM
Unsloth: Training lm_head in mixed precision to save VRAM
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
[Rank 3] start!
[Rank 3] model dir: /kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1
[Rank 3] config: {'model_path': '', 'out_dir': '/kaggle/inference_outputs', 'lora_seed': 42, 'train_aug_seed': 1, 'n_train_aug': 16, 'num_epochs': 1, 'learning_rate': 5e-05, 'eval_aug_seed': 2, 'n_eval_aug': 2, 'min_prob': 0.2, 'dfs_window': 540.0, 'task_cap': 1200.0, 'score_seed_off': 0, 'decode_batch': 4}
==((====))==  Unsloth 2025.9.7: Fast Qwen3 patching. Transformers: 4.55.4.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = True]
 "-____-"     

Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.00s/it]


Unsloth: Training embed_tokens in mixed precision to save VRAM
Unsloth: Training lm_head in mixed precision to save VRAM
Unsloth: Training embed_tokens in mixed precision to save VRAM
Unsloth: Training lm_head in mixed precision to save VRAM
[Rank 1] allocated 13011MB for training
[Rank 1] training stats for puzzle 0934a4d8: TrainOutput(global_step=128, training_loss=0.004536581691354513, metrics={'train_runtime': 384.3957, 'train_samples_per_second': 0.333, 'train_steps_per_second': 0.333, 'total_flos': 1.232880211132416e+16, 'train_loss': 0.004536581691354513, 'epoch': 1.0})
[Rank 1] decoding ['0934a4d8_0.permute4150723698.ex0312', '0934a4d8_0.permute9302675184.ex1302', '0934a4d8_0.rot90.rot90.permute5386701942.ex0213', '0934a4d8_0.rot90.rot90.permute8579032416.ex3201']
[Rank 1] scoring 0934a4d8_0.permute4150723698.ex0312 #0
[Rank 1] scoring 0934a4d8_0.permute4150723698.ex0312 #1
[Rank 1] scoring 0934a4d8_0.permute9302675184.ex1302 #0
[Rank 1] decoding ['0934a4d8_0.rot90.permute42816

In [8]:
# ---------------------------------------------------------------------------
# Phase 2 — adaptive use of any leftover time (skipped if < 20 min remain):
#   (a) catch-up: tasks the primary pass never reached (same LB-proven config)
#   (b) deep: outputs with < 2 unique candidates get a wider search
#       (different LoRA/aug seeds, 24 views, DFS threshold 0.1, longer windows)
#   Primary results are never modified; phase-2 candidates are pooled and
#   vote together under the same score_kgmon selection.
# ---------------------------------------------------------------------------
import os, sys, json, time, subprocess
from arc_loader import ArcDataset
from arc_decoder import ArcDecoder

def remaining():
    return global_end_time - time.time()

test_path = ("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json" if RERUN
             else "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json")
data = ArcDataset.from_file(test_path)
keys_in_scope = data.keys if RERUN else [k for k in data.keys if k in os.getenv("ARC_DEBUG_KEYS", "0934a4d8,36a08778,981571dc,aa4ec2a5").split(",")]

dec = ArcDecoder(data.split_multi_replies(), n_guesses=2)
dec.load_decoded_results("/kaggle/inference_outputs")
stats = dec.candidate_stats()

unprocessed, starved = [], {}
for k in keys_in_scope:
    n_out = len(data.queries[k]["test"])
    outs = [f"{k}_{i}" for i in range(n_out)]
    if not any(o in stats for o in outs):
        unprocessed.append(k)
        continue
    n_starved = sum(1 for o in outs if stats.get(o, {"unique": 0})["unique"] < 2)
    if n_starved:
        starved[k] = n_starved

print(f"[phase-2] unprocessed={len(unprocessed)} starved_tasks={len(starved)} remaining={remaining()/60:.1f} min")
json.dump({"unprocessed": unprocessed, "starved": starved}, open("/kaggle/working/phase2_plan.json", "w"))

base_env = dict(os.environ, UNSLOTH_DISABLE_STATISTICS="1", TRITON_PTXAS_PATH="/usr/local/cuda/bin/ptxas",
                OMP_NUM_THREADS="12", PYTHONHASHSEED="0")

def run_phase(name, keys, env_overrides, out_dir):
    if not keys or remaining() < 20 * 60:
        print(f"[phase-2:{name}] skipped (keys={len(keys)}, remaining={remaining()/60:.1f} min)")
        return
    keys_file = f"/kaggle/working/keys_{name}.json"
    json.dump(keys, open(keys_file, "w"))
    env = dict(base_env, ARC_OUT_DIR=out_dir, **{k: str(v) for k, v in env_overrides.items()})
    t = time.time()
    rc = subprocess.call([sys.executable, "starter.py", "--end-time", f"{global_end_time}", "--keys-file", keys_file, "--order", "file"], env=env)
    print(f"[phase-2:{name}] rc={rc} keys={len(keys)} took {(time.time()-t)/60:.1f} min; remaining {remaining()/60:.1f} min")

# (a) catch-up with the primary configuration (cheap-first order preserved from phase 1)
from starter import estimated_work
unprocessed = sorted(unprocessed, key=lambda k: estimated_work(data.queries[k]))
run_phase("catchup", unprocessed, {}, "/kaggle/inference_outputs")

# (b) deep pass on starved outputs: most starved outputs first, then cheapest
deep_keys = sorted(starved, key=lambda k: (-starved[k], estimated_work(data.queries[k])))
if deep_keys and remaining() >= 20 * 60:
    per_task = max(600.0, min(2400.0, (remaining() - 300) * 4.0 / len(deep_keys)))
    deep_cfg = dict(ARC_LORA_SEED=137, ARC_TRAIN_AUG_SEED=17, ARC_EVAL_AUG_SEED=29, ARC_N_EVAL_AUG=3,
                    ARC_MIN_PROB=0.1, ARC_DFS_WINDOW=600, ARC_TASK_CAP=int(per_task), ARC_SCORE_SEED_OFFSET=7)
    print(f"[phase-2:deep] per-task cap {per_task:.0f}s, config {deep_cfg}")
    run_phase("deep", deep_keys, deep_cfg, "/kaggle/inference_outputs_deep")


*** Loaded 74 shards / 88 samples from /kaggle/inference_outputs (skipped 0)
[phase-2] unprocessed=0 starved_tasks=1 remaining=29.2 min
[phase-2:catchup] skipped (keys=0, remaining=29.2 min)
[phase-2:deep] per-task cap 2400s, config {'ARC_LORA_SEED': 137, 'ARC_TRAIN_AUG_SEED': 17, 'ARC_EVAL_AUG_SEED': 29, 'ARC_N_EVAL_AUG': 3, 'ARC_MIN_PROB': 0.1, 'ARC_DFS_WINDOW': 600, 'ARC_TASK_CAP': 2400, 'ARC_SCORE_SEED_OFFSET': 7}
[starter] 1 tasks, 4 workers, order=file, budget=29.1 min, test_path=/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
[Rank 0] start!
[Rank 0] model dir: /kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1
[Rank 0] config: {'model_path': '', 'out_dir': '/kaggle/inference_outputs_deep', 'lora_seed': 137, 'train_aug_seed': 17, 'n_train_aug': 16, 'num_epochs': 1, 'learning_rate

Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.02s/it]


Unsloth: Training embed_tokens in mixed precision to save VRAM
Unsloth: Training lm_head in mixed precision to save VRAM
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
[Rank 1] start!
[Rank 1] model dir: /kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1
[Rank 1] config: {'model_path': '', 'out_dir': '/kaggle/inference_outputs_deep', 'lora_seed': 137, 'train_aug_seed': 17, 'n_train_aug': 16, 'num_epochs': 1, 'learning_rate': 5e-05, 'eval_aug_seed': 29, 'n_eval_aug': 3, 'min_prob': 0.1, 'dfs_window': 600.0, 'task_cap': 2400.0, 'score_seed_off': 7, 'decode_batch': 4}
==((====))==  Unsloth 2025.9.7: Fast Qwen3 patching. Transformers: 4.55.4.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = True]
 "-___

Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.02s/it]


Unsloth: Training embed_tokens in mixed precision to save VRAM
Unsloth: Training lm_head in mixed precision to save VRAM
[Rank 1] done!
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
[Rank 2] start!
[Rank 2] model dir: /kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1
[Rank 2] config: {'model_path': '', 'out_dir': '/kaggle/inference_outputs_deep', 'lora_seed': 137, 'train_aug_seed': 17, 'n_train_aug': 16, 'num_epochs': 1, 'learning_rate': 5e-05, 'eval_aug_seed': 29, 'n_eval_aug': 3, 'min_prob': 0.1, 'dfs_window': 600.0, 'task_cap': 2400.0, 'score_seed_off': 7, 'decode_batch': 4}
==((====))==  Unsloth 2025.9.7: Fast Qwen3 patching. Transformers: 4.55.4.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2

Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.02s/it]


Unsloth: Training embed_tokens in mixed precision to save VRAM
Unsloth: Training lm_head in mixed precision to save VRAM
[Rank 2] done!
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
[Rank 3] start!
[Rank 3] model dir: /kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1
[Rank 3] config: {'model_path': '', 'out_dir': '/kaggle/inference_outputs_deep', 'lora_seed': 137, 'train_aug_seed': 17, 'n_train_aug': 16, 'num_epochs': 1, 'learning_rate': 5e-05, 'eval_aug_seed': 29, 'n_eval_aug': 3, 'min_prob': 0.1, 'dfs_window': 600.0, 'task_cap': 2400.0, 'score_seed_off': 7, 'decode_batch': 4}
==((====))==  Unsloth 2025.9.7: Fast Qwen3 patching. Transformers: 4.55.4.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2

Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.02s/it]


Unsloth: Training embed_tokens in mixed precision to save VRAM
Unsloth: Training lm_head in mixed precision to save VRAM
[Rank 3] done!
Unsloth: Training embed_tokens in mixed precision to save VRAM
Unsloth: Training lm_head in mixed precision to save VRAM
[Rank 0] allocated 13150MB for training
[Rank 0] training stats for puzzle 981571dc: TrainOutput(global_step=128, training_loss=0.0001164611749118194, metrics={'train_runtime': 842.7951, 'train_samples_per_second': 0.152, 'train_steps_per_second': 0.152, 'total_flos': 2.385848559992832e+16, 'train_loss': 0.0001164611749118194, 'epoch': 1.0})
[Rank 0] decoding ['981571dc_0.permute7146980235.ex230', '981571dc_0.permute8617029435.ex103', '981571dc_0.permute9852317064.ex023', '981571dc_0.rot90.rot90.permute1395208647.ex230']
[Rank 0] scoring 981571dc_0.permute7146980235.ex230 #0
[Rank 0] decoding ['981571dc_0.rot90.rot90.permute5038426917.ex230', '981571dc_0.rot90.rot90.permute5897401326.ex023', '981571dc_0.rot90.permute7205186349.ex302'

In [9]:
# ---------------------------------------------------------------------------
# Final selection: pool primary (+catch-up) and deep candidates, score_kgmon
# (vote count - mean augmented NLL), top-2 -> attempt_1/attempt_2.
# Outputs without any candidate fall back to identity / [[0]].
# ---------------------------------------------------------------------------
import os, json, hashlib
import numpy as np
from arc_loader import ArcDataset
from arc_decoder import ArcDecoder

test_path = ("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json" if RERUN
             else "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json")
data = ArcDataset.from_file(test_path)
if not RERUN:
    data = data.load_replies("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_solutions.json")

decoder = ArcDecoder(data.split_multi_replies(), n_guesses=2)
decoder.load_decoded_results("/kaggle/inference_outputs")
decoder.load_decoded_results("/kaggle/inference_outputs_deep", run_name=".deep")

selected = decoder.run_selection_algo()
submission = data.get_submission(selected)

# fallbacks + schema hardening
n_fallback = 0
for k in data.keys:
    for i, t in enumerate(data.queries[k]["test"]):
        e = submission[k][i]
        for a in ("attempt_1", "attempt_2"):
            g = e.get(a)
            ok = isinstance(g, list) and len(g) > 0 and all(isinstance(r, list) and len(r) == len(g[0]) and len(r) > 0 for r in g)
            if not ok or g == [[0]]:
                if a == "attempt_1":
                    e[a] = [[int(x) for x in row] for row in t["input"]]
                    n_fallback += 1
                else:
                    e[a] = e[a] if (ok and g != [[0]]) else [[0]]
        if e["attempt_2"] == e["attempt_1"]:
            e["attempt_2"] = [[0]]
        e["attempt_1"] = [[int(x) for x in row] for row in e["attempt_1"]]
        e["attempt_2"] = [[int(x) for x in row] for row in e["attempt_2"]]

with open("/kaggle/working/submission.json", "w") as f:
    json.dump(submission, f)
print(f"*** wrote submission.json: {len(submission)} tasks, {sum(len(v) for v in submission.values())} outputs, {n_fallback} identity fallbacks, "
      f"sha256={hashlib.sha256(open('/kaggle/working/submission.json','rb').read()).hexdigest()[:12]}")

# integrity re-check
chk = json.load(open("/kaggle/working/submission.json"))
assert set(chk) == set(data.keys), "missing task ids"
for k in data.keys:
    assert len(chk[k]) == len(data.queries[k]["test"]), f"wrong #outputs for {k}"
    for e in chk[k]:
        assert "attempt_1" in e and "attempt_2" in e
print("*** submission schema OK")

if not RERUN:
    decoder.benchmark_selection_algos()
    print("*** Reload score (tasks):", data.validate_submission(chk), "of", len([k for k in data.keys if any(f'{k}_{i}' in decoder.decoded_results for i in range(9))]), "attempted")


*** Load solutions from '/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_solutions.json'...
*** Loaded 74 shards / 88 samples from /kaggle/inference_outputs (skipped 0)
*** Loaded 24 shards / 24 samples from /kaggle/inference_outputs_deep (skipped 0)
*** Generating submission for 5 outputs...
*** wrote submission.json: 120 tasks, 172 outputs, 167 identity fallbacks, sha256=2b848186d95f
*** submission schema OK
*** Benchmark selection algorithms...
ALL_CORRECT: 0.00284 -  0.00082 30x30 [981571dc_0.rot90.rot90.permute8579032416.ex201.out0]
ALL_CORRECT: 0.00110 -  0.00082 30x30 [981571dc_0.rot90.rot90.permute5386701942.ex213.out0]
ALL_CORRECT: 0.00023 -  0.00082 30x30 [981571dc_0.transpose.rot90.permute4907281356.ex210.out0]
ALL_CORRECT: 0.00012 -  0.00082 30x30 [981571dc_0.permute9302675184.ex302.out0]
ALL_CORRECT: 0.00002 -  0.00082 30x30 [981571dc_0.rot90.permute4950812673.ex013.out0]
ALL_CORRECT: 0.00009 -  0.00082 30x30 [981571dc_0.transpose.rot90.rot90.rot90.p